# build model

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass ,field

class MLP(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dim=512,
    ):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.layers(x)
    
class NHiTSBlock(nn.Module):
    def __init__(self,
                 input_length, 
                 num_channels,
                 forecast_length,
                 n_freq_downsample,
                 hidden_size,
                 pooling_kernel_size=2,
                 pooling_stride = 2,
                 ):
        super(NHiTSBlock, self).__init__()
        
        pooled_length = input_length // pooling_kernel_size
        self.input_length = input_length
        self.forecast_length = forecast_length
        self.num_channels = num_channels
        mlp_input_size = pooled_length*num_channels
        
        self.output_size = input_length + forecast_length//n_freq_downsample
        self.basis = _IdentityBasis(input_length,forecast_length)
        self.mlp = MLP(mlp_input_size, self.output_size, hidden_size)
        self.poolinglayer = nn.MaxPool1d(kernel_size=pooling_kernel_size,
                                         stride=pooling_stride,
                                         ceil_mode=True)

    def forward(self, x):
        B,T,C = x.shape #(B,T,C) = (batch_size, time_steps, channels)
        x = x.permute(0,2,1) #(B,C,T)
        x = self.poolinglayer(x) # (B,C,T//2)
        x = x.permute(0,2,1).contiguous().flatten(-2) #(B,T*C//2)
        theta = self.mlp(x) #(B,output_dim)
        backcast ,forecast= self.basis(theta)
        return backcast, forecast
    
class _IdentityBasis(nn.Module):
    def __init__(self, backcast_length, forecast_length,iterpolation_mode='linear'):
        super(_IdentityBasis, self).__init__()
        self.backcast_length = backcast_length
        self.forecast_length = forecast_length
        self.interpolation_mode = iterpolation_mode

    def forward(self, theta:torch.Tensor):# (batch_size, backcast_length + forecast_length)
        # theta shape: (batch_size, backcast_length + forecast_length)
        backcast = theta[:, :self.backcast_length]
        forecast = theta[:, self.backcast_length:]
        forecast = forecast.unsqueeze(1) # (B,1,L)
        forecast = F.interpolate(
            forecast,
            size=self.forecast_length,
            mode=self.interpolation_mode
            )  # Add a new dimension for channels
        forecast = forecast.squeeze(1) #(B,L)
        return backcast, forecast
    
class TemporalNorm(nn.Module):
    def __init__(self, eps=1e-5):
        super(TemporalNorm, self).__init__()
        self.eps = eps

    def forward(self, x):
        # x shape: (batch_size, time_steps, channels)
        mean = x.mean(dim=1, keepdim=True)
        
        std = torch.sqrt(
        x.var(dim=1, keepdim=True, unbiased=False) + self.eps
)
        return (x - mean) / (std + self.eps),mean,std
    
    def inverse(self, x, mean, std):
        return x * std  + mean


@dataclass# 原论文的config设置
class NHITSConfig:
    input_length :int = 60
    num_channels:int = 1
    num_blocks:int =3
    hidden_size:int =512
    forecast_length:int = 12
    n_freq_downsample: list[int] = field(
        default_factory=lambda: [4, 2, 1])
    pooling_kernel_size: list[int] = field(
        default_factory=lambda: [2, 2, 1])
    pooling_stride: list[int] = field(
        default_factory=lambda: [2, 2, 1])

class NHITS(nn.Module):
    def __init__(self,config:NHITSConfig):
        super().__init__()
        self.num_blocks = config.num_blocks
        blocks = []
        for i in range(config.num_blocks):
            blocks.append(NHiTSBlock(
                input_length=config.input_length,
                num_channels=config.num_channels,
                hidden_size=config.hidden_size,
                forecast_length=config.forecast_length,
                n_freq_downsample=config.n_freq_downsample[i],
                pooling_kernel_size=config.pooling_kernel_size[i],
                pooling_stride=config.pooling_stride[i],
                ))
        self.NHiTS = nn.ModuleList(blocks)
    def forward(self,x:torch.Tensor):
        residual = x.clone() #b,t,c
        
        forecast = torch.zeros(
            x.size(0),
            self.NHiTS[0].forecast_length,
            device=x.device,
            dtype=x.dtype
        )
        
        for b in self.NHiTS:
            backcast, block_forecast = b(residual) #b,input_length    b,forecast_length
            backcast = backcast.unsqueeze(-1)
            residual=residual-backcast
            forecast=forecast+block_forecast
        return forecast



# create dataloader

In [4]:
import torch
from torch.utils.data import Dataset
class TimeSeriesDataset(Dataset):
    def __init__(
        self,
        data,
        input_length,
        forecast_length,
        stride=1,):
        super().__init__()
        
        self.data = torch.as_tensor(data,dtype=torch.float32)
        assert self.data.ndim == 2 #(b,1)
        
        self.input_length = input_length
        self.forecast_length = forecast_length
        self.stride =stride
        
        self.length = (len(self.data)-input_length-forecast_length)//stride+1
        if self.length <=0:
            raise ValueError(
                "Time series is too short for the given "
                "input_length and forecast_length."
            )
            
    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        start = idx * self.stride

        x_start = start
        x_end = start + self.input_length

        y_start = x_end
        y_end = y_start + self.forecast_length

        x = self.data[x_start:x_end]
        y = self.data[y_start:y_end]

        return x, y

# create data

In [64]:
import numpy as np
from torch.utils.data import DataLoader
import pandas as pd
data = pd.read_csv(r"data/ETTh1.csv")

otdata = data["OT"]
ot = torch.Tensor(otdata.to_numpy()).reshape(-1,1)
# data = data.reshape(-1, 1)
#(1000,1)
train_size = int(ot.size(0)*0.8)
ottrain = ot[:train_size]
otval = ot[train_size:]

ottraindataset = TimeSeriesDataset(
    data=ottrain,
    input_length=60,
    forecast_length=12,
)

otvaldataset = TimeSeriesDataset(
    data=otval,
    input_length=60,
    forecast_length=12,
)

trainloader = DataLoader(
    ottraindataset,
    batch_size=32,
    shuffle=True,
)

valloader = DataLoader(
    otvaldataset,
    batch_size=32,
    shuffle=True,
)


In [65]:
import torch.optim as optim

device = 'cuda:0'
config = NHITSConfig()
model = NHITS(config=config).to(device)
# model = NHITS(config=config)
opt = optim.AdamW(model.parameters())

In [ ]:
# torch.autograd.set_detect_anomaly(False)

for epoch in range(1000):
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for x,y in trainloader:
        x = x.to(device)
        y = y.to(device)

        ypred = model(x)
        target = y.squeeze(-1)

        loss = F.l1_loss(ypred, target)

        opt.zero_grad()
        loss.backward()
        opt.step()

        epoch_loss += loss.item()
        n_batches += 1
    epoch_loss /= n_batches
    
    if epoch %50 ==0:
        model.eval()
    model.eval()

    val_loss = 0.0
    n_val = 0

    with torch.no_grad():
        for xval, yval in valloader:
            xval = xval.to(device)
            yval = yval.to(device)

            ypred = model(xval)
            target = yval.squeeze(-1)

            loss = F.l1_loss(ypred, target)

            val_loss += loss.item()
            n_val += 1

            val_loss /= n_val
        model.train()
        print(f"Epoch {epoch:03d} | train-MAE = {epoch_loss:.6f} | val-MAE = {loss:.6f}")

Epoch 000 | train-MAE = 1.732693 | val-MAE = 0.862749
Epoch 050 | train-MAE = 0.717176 | val-MAE = 1.237105


2.6.0+cu124
12.4
True
Quadro RTX 8000


In [1]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
import torch

horizon = 12
models = [
    NHITS(
        input_size=5*horizon, #60
        h=horizon,
        max_steps=10,
    )
]
nf = NeuralForecast(models=models, freq='H')

lit_model = nf.models[0]

# 打印完整模型结构
print("=====完整模型=====")
print(lit_model)

print("\n====遍历所有子模块====")
for name, module in lit_model.named_modules():
    print(f"{name:<40} | {module}")


# 重点看核心组件，对应论文
print("\n====blocks列表====")
print(lit_model.blocks)

# 取出第一个block
block0 = lit_model.blocks[0]
print("\n====第一个block====")
print(block0)

# Basis插值模块，就是论文附录Haar分段常数基！！
basis = block0.basis
print("\n====Basis插值模块（核心）====")
print(basis)


/home/stu001/miniconda3/envs/nf_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-14 20:30:09,784	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-09-14 20:30:10,363	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Seed set to 1


=====完整模型=====
NHITS

====遍历所有子模块====
                                         | NHITS
loss                                     | MAE()
hist_cat_embeddings                      | ModuleList()
futr_cat_embeddings                      | ModuleList()
stat_cat_embeddings                      | ModuleList()
padder_train                             | ConstantPad1d(padding=(0, 12), value=0.0)
scaler                                   | TemporalNorm()
blocks                                   | ModuleList(
  (0): NHITSBlock(
    (pooling_layer): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (layers): Sequential(
      (0): Linear(in_features=30, out_features=512, bias=True)
      (1): Linear(in_features=512, out_features=512, bias=True)
      (2): ReLU()
      (3): Linear(in_features=512, out_features=512, bias=True)
      (4): ReLU()
      (5): Linear(in_features=512, out_features=512, bias=True)
      (6): ReLU()
      (7): Linear(in_features=512, out_features=6